# Применение (inference) и оценка модели FinGPT

In [ ]:
#### Рабочая установка !!!

#  Шаг 1: Полная очистка окружения
# Закройте все программы, использующие Python (Jupyter, VS Code). Затем в терминале (обязательно от имени администратора) выполните:

# bash
# pip uninstall bitsandbytes peft transformers accelerate torch torchvision torchaudio -y
# pip cache purge
# Шаг 2: Установка PyTorch 2.5.1
# Установите PyTorch с поддержкой CUDA 12.4 (максимальная стабильная версия для Windows):

# bash
# pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
# Проверка:

# python
# import torch
# print(torch.cuda.is_available())  # Должно быть True
# print(torch.__version__)          # Должно быть 2.5.1+cu124
# Шаг 3: Установка совместимых версий
# Выполните установку в указанном порядке:

# bash
# pip install bitsandbytes==0.44.0
# pip install accelerate==1.1.1
# pip install transformers==4.46.3
# pip install peft==0.14.0
# pip install sentencepiece
# Шаг 4: Проверка bitsandbytes
# Запустите диагностику:

# bash
# python -m bitsandbytes
# В выводе должна быть строка SUCCESS. Если её нет, библиотека не видит GPU.

# Шаг 5: Установка переменной окружения
# Перед загрузкой модели укажите версию CUDA (12.4 для PyTorch):

# bash
# set BNB_CUDA_VERSION=124
# или
# import os
# os.environ['BNB_CUDA_VERSION'] = '124'
# print(os.environ.get('BNB_CUDA_VERSION'))

In [1]:
import torch
print(torch.cuda.is_available())  # Должно быть True
print(torch.__version__) 

True
2.5.1+cu124


In [1]:
#for the `load_in_8bit=True` error
# !pip install protobuf transformers==4.32.0 cpm_kernels torch>=2.0 gradio mdtex2html sentencepiece accelerate

In [2]:
import datasets
import torch
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast
from peft import PeftModel  # 0.5.0
import re

E:\alena\fin_proj\fin_proj\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
E:\alena\fin_proj\fin_proj\lib\site-packages\transformers\utils\hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
import transformers
print(transformers.is_bitsandbytes_available())

True


#### Загрузка  модели

In [4]:
from transformers import BitsAndBytesConfig

base_model = "NousResearch/Llama-2-7b-hf"
peft_model = "FinGPT/fingpt-mt_llama2-7b_lora"
tokenizer = LlamaTokenizerFast.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
# model = LlamaForCausalLM.from_pretrained(base_model, trust_remote_code=True, device_map = "cuda:0",)


bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,  # 8-bit квантование
    llm_int8_threshold=6.0
)

model = LlamaForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,  # вместо load_in_8bit=True
    device_map="auto",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(model, peft_model)
model = model.eval()

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Loading checkpoint shards: 100%|██████████| 2/2 [03:11<00:00, 95.58s/it] 


#### Тестирование модели на пользовательских примерах

In [7]:
prompt = [
'''Instruction: 'Please extract entities and their types from the input sentence, entity types should be chosen from {person/organization/location}.',
Input: 'Apple Inc. reported record profits of $100 billion.,
Answer: '''
]

tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512)
res = model.generate(**tokens, max_length=512)
res_sentences = [tokenizer.decode(i) for i in res]
out_text = [o.split("Answer: ")[1] for o in res_sentences]

for text in out_text:
    print(text)



 Apple Inc is an organization.</s>


In [8]:
prompt = [
'''Instruction: 'Please extract entities and their types from the input sentence, entity types should be chosen from {person/organization/location}.',
Input: 'While SpaceX has taken an unusual approach to its offering, setting up access for retail investors through brokerage firms at a level atypical in new deals typically dominated by institutions, the NASA fund is another alternative for investors to gain access to Elon Musk’s rocket company.',
Answer: '''
]

tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512)
res = model.generate(**tokens, max_length=512)
res_sentences = [tokenizer.decode(i) for i in res]
out_text = [o.split("Answer: ")[1] for o in res_sentences]

for text in out_text:
    print(text)



 NASA is an organization.</s>


In [9]:


prompt = [
'''Instruction: 'Please extract entities and their types from the input sentence, entity types should be chosen from {person/organization/location}.',
Input: 'The borrower Sarah Johnson holds deposits of $50000',
Answer: '''
]

tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512)
res = model.generate(**tokens, max_length=512)
res_sentences = [tokenizer.decode(i) for i in res]
out_text = [o.split("Answer: ")[1] for o in res_sentences]

for text in out_text:
    print(text)

 borrower is a person, Sarah Johnson is a person.</s>


#### Парсинг ответов и применение на пользовательских примерах

In [10]:
import re

In [12]:
def parse_response(answer):
    if not answer:
        return []
    
    entities = []
    
    answer = answer.rstrip('.')
    
    parts = answer.split(', ')
    
    for part in parts:
        part = part.strip()
        if not part:
            continue
        
        if ' is a ' in part:
            entity_text, type_text = part.split(' is a ', 1)
            type_text = type_text.strip()
        elif ' is an ' in part:
            entity_text, type_text = part.split(' is an ', 1)
            type_text = type_text.strip()
        else:
            continue
        
        entity_text = entity_text.strip()
        type_text = type_text.strip().lower()
        
        # Нормализуем тип
        if type_text == 'person':
            entity_type = 'PERSON'
        elif type_text == 'organization':
            entity_type = 'ORGANIZATION'
        elif type_text == 'location':
            entity_type = 'LOCATION'
        else:
            continue
        
        if len(entity_text) < 2:
            continue
        
        entity_text = entity_text.rstrip('.')
        
        entities.append(f"{entity_type}: {entity_text}")
    
    return entities

def extract_entities(text):
    prompt = f"""Instruction: 'Please extract entities and their types from the input sentence, entity types should be chosen from {{person/organization/location}}.',\nInput: '{text}',\nAnswer: """
    
    tokens = tokenizer(prompt, return_tensors='pt', padding=True, 
                       max_length=1024, truncation=True) 
    tokens = {k: v.to(model.device) for k, v in tokens.items()}  
    
    with torch.no_grad():
        res = model.generate(**tokens, max_new_tokens=256, temperature=0.1,
                            pad_token_id=tokenizer.eos_token_id)
    
    res_text = tokenizer.decode(res[0], skip_special_tokens=True)
    
    if "Answer:" in res_text:
        answer = res_text.split("Answer:")[1].strip()
    else:
        answer = res_text
    
    entities = parse_response(answer)
    return entities, answer


def process_sentence(text):

    prompt = f"""Instruction: 'Please extract entities and their types from the input sentence, entity types should be chosen from {{person/organization/location}}.',
Input: '{text}',
Answer: """
    
    # Токенизация
    tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512, truncation=True)
    tokens = {k: v.to(model.device) for k, v in tokens.items()}
    
    # Генерация
    with torch.no_grad():
        res = model.generate(
            **tokens, 
            max_new_tokens=256, 
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False
        )
    
    # декодирование
    res_text = tokenizer.decode(res[0], skip_special_tokens=True)
    
    # извлечение ответа
    if "Answer:" in res_text:
        answer = res_text.split("Answer:")[1].strip()
    else:
        answer = res_text
    
    return parse_response(answer)

examples = [
    "Apple Inc. reported record profits of $100 billion.",
    "While SpaceX has taken an unusual approach to its offering, setting up access for retail investors through brokerage firms at a level atypical in new deals typically dominated by institutions, the NASA fund is another alternative for investors to gain access to Elon Musk's rocket company.",
    "Mutual fund manager and billionaire Ron Baron, a long-time Tesla and SpaceX investor, owns the rocket company through his First Principles fund (RONB)",
    "John Smith works at Google in New York",
    "The borrower Sarah Johnson holds deposits of $50000",
    "Microsoft CEO Satya Nadella visited London yesterday",
    "This LOAN AND SECURITY AGREEMENT dated January 27, 1999, between SILICON VALLEY BANK ( Bank ), a California - chartered bank with its principal place of business at 3003 Tasman Drive, Santa Clara, California 95054 with a loan production office located at 40 William St., Ste."
]

for example in examples:
    entities = process_sentence(example)
    print(f"Текст: {example}")
    print(f"Сущности: {entities}")
    print()

Текст: Apple Inc. reported record profits of $100 billion.
Сущности: ['ORGANIZATION: Apple Inc']

Текст: While SpaceX has taken an unusual approach to its offering, setting up access for retail investors through brokerage firms at a level atypical in new deals typically dominated by institutions, the NASA fund is another alternative for investors to gain access to Elon Musk's rocket company.
Сущности: ['ORGANIZATION: SpaceX', 'ORGANIZATION: NASA']

Текст: Mutual fund manager and billionaire Ron Baron, a long-time Tesla and SpaceX investor, owns the rocket company through his First Principles fund (RONB)
Сущности: ['PERSON: Ron Baron', 'ORGANIZATION: SpaceX', 'ORGANIZATION: Tesla', 'PERSON: Ron Baron']

Текст: John Smith works at Google in New York
Сущности: ['PERSON: John Smith', 'ORGANIZATION: Google']

Текст: The borrower Sarah Johnson holds deposits of $50000
Сущности: []

Текст: Microsoft CEO Satya Nadella visited London yesterday
Сущности: ['ORGANIZATION: Microsoft', 'PERSON: Saty

In [13]:
def load_bio_data(file_path):
    sentences = []
    labels = []
    current_sentence = []
    current_labels = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: 
                if current_sentence:
                    sentences.append(current_sentence)
                    labels.append(current_labels)
                    current_sentence = []
                    current_labels = []
            else:
                parts = line.split()
                if len(parts) == 2:
                    current_sentence.append(parts[0])
                    current_labels.append(parts[1])
    
    if current_sentence:
        sentences.append(current_sentence)
        labels.append(current_labels)
    
    return sentences, labels

def convert_bio_to_entities(tokens, bio_labels):
    entities = []
    current_entity = []
    current_type = None
    
    type_mapping = {'PER': 'PERSON', 'ORG': 'ORGANIZATION', 'LOC': 'LOCATION'}
    
    for token, label in zip(tokens, bio_labels):
        if label == 'O':
            if current_entity:
                entities.append(f"{current_type}: {' '.join(current_entity)}")
                current_entity = []
                current_type = None
        elif label.startswith('B-'):
            if current_entity:
                entities.append(f"{current_type}: {' '.join(current_entity)}")
            bio_type = label[2:]
            current_type = type_mapping.get(bio_type, bio_type)
            current_entity = [token]
        elif label.startswith('I-') and current_entity:
            current_entity.append(token)
    
    if current_entity:
        entities.append(f"{current_type}: {' '.join(current_entity)}")
    
    return entities


from collections import Counter

def calculate_metrics(true_entities, pred_entities):
    true_counter = Counter(true_entities)
    pred_counter = Counter(pred_entities)
    
    all_entities = set(true_counter.keys()) | set(pred_counter.keys())
    
    tp = 0
    for ent in all_entities:
        tp += min(true_counter.get(ent, 0), pred_counter.get(ent, 0))
    
    fp = sum(pred_counter.values()) - tp
    fn = sum(true_counter.values()) - tp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'fn': fn
    }


file_path = './data/test_mod.txt'
sentences, true_labels = load_bio_data(file_path)
print(f"Загружено {len(sentences)} предложений")

# test_size = min(50, len(sentences))
test_size = len(sentences)
sentences = sentences[:test_size]
true_labels = true_labels[:test_size]

print(f"Будет обработано {test_size} предложений")

all_true_entities = []
all_pred_entities = []
failed_cases = []

print("\nНачинаем оценку...")

for i, (tokens, bio_labels) in enumerate(zip(sentences, true_labels)):
    text = ' '.join(tokens)
    
    # получаем истинные сущности из BIO
    true_entities = convert_bio_to_entities(tokens, bio_labels)
    
    # получаем предсказания модели
    try:
        pred_entities, model_answer = extract_entities(text)
    except Exception as e:
        pred_entities = []
        model_answer = f"ERROR: {e}"
        failed_cases.append(i)
    
    all_true_entities.extend(true_entities)
    all_pred_entities.extend(pred_entities)
    
    # выводим примеры
    if i < 10 or (i + 1) % 10 == 0:
        print(f"\nПример {i+1}")
        print(f"Текст: {text}")
        print(f"Истинные сущности: {true_entities}")
        print(f"Предсказанные: {pred_entities}")
        if model_answer and model_answer != "ERROR":
            print(f"Ответ модели: {model_answer}")

overall_metrics = calculate_metrics(all_true_entities, all_pred_entities)

print("\nРезультаты оценки")

print(f"\n Статистика:")
print(f"Всего истинных сущностей: {len(all_true_entities)}")
print(f"Всего предсказано сущностей: {len(all_pred_entities)}")
print(f"\n Метрики:")
print(f"\nPrecision: {overall_metrics['precision']:.4f} ({overall_metrics['precision']*100:.2f}%)")
print(f"Recall: {overall_metrics['recall']:.4f} ({overall_metrics['recall']*100:.2f}%)")
print(f"F1-Score: {overall_metrics['f1']:.4f} ({overall_metrics['f1']*100:.2f}%)")

print("Метрики по типам сущностей")

entity_types = ['PERSON', 'ORGANIZATION', 'LOCATION']

for entity_type in entity_types:
    true_type = [e for e in all_true_entities if e.startswith(entity_type)]
    pred_type = [e for e in all_pred_entities if e.startswith(entity_type)]
    
    metrics = calculate_metrics(true_type, pred_type)
    
    print(f"\n{entity_type}:")
    print(f"Истинных: {len(true_type)}, Предсказано: {len(pred_type)}")
    print(f"Precision: {metrics['precision']:.4f} ({metrics['precision']*100:.2f}%)")
    print(f"Recall: {metrics['recall']:.4f} ({metrics['recall']*100:.2f}%)")
    print(f"F1-Score: {metrics['f1']:.4f} ({metrics['f1']*100:.2f}%)")

# Сохраняем результаты в файл
import json
from datetime import datetime

results = {
    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'test_size': test_size,
    'true_entities': all_true_entities,
    'pred_entities': all_pred_entities,
    'metrics': overall_metrics,
    'metrics_by_type': {}
}

for entity_type in entity_types:
    true_type = [e for e in all_true_entities if e.startswith(entity_type)]
    pred_type = [e for e in all_pred_entities if e.startswith(entity_type)]
    results['metrics_by_type'][entity_type] = calculate_metrics(true_type, pred_type)

with open('ner_evaluation_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Результаты сохранены в файл 'ner_evaluation_results.json'")

Загружено 303 предложений
Будет обработано 303 предложений

Начинаем оценку...

Пример 1
Текст: Subordinated Loan Agreement - Silicium de Provence SAS and Evergreen Solar Inc . 7 - December 2007 [ HERBERT SMITH LOGO ] ................................ 2007 SILICIUM DE PROVENCE SAS and EVERGREEN SOLAR , INC .
Истинные сущности: ['ORGANIZATION: Silicium de Provence SAS', 'ORGANIZATION: Evergreen Solar Inc', 'PERSON: HERBERT SMITH', 'ORGANIZATION: SILICIUM DE PROVENCE SAS', 'ORGANIZATION: EVERGREEN SOLAR']
Предсказанные: ['ORGANIZATION: SAS', 'ORGANIZATION: SAS', 'ORGANIZATION: SAS', 'ORGANIZATION: SAS']
Ответ модели: SAS is an organization, SAS is an organization, SAS is an organization, SAS is an organization.

Пример 2
Текст: SUBORDINATED LOAN AGREEMENT HERBERT SMITH LLP Page 1 of 12 7 - December 2007 TABLE OF CONTENTS Clause Headings Page 1 .
Истинные сущности: ['PERSON: HERBERT SMITH']
Предсказанные: ['PERSON: HBERT SMITH']
Ответ модели: HBERT SMITH is a person.

Пример 3
Текст: INTER